# 🔥 Обновление данных сайта ГВС-Ярославль

Конвертирует Excel → `data.json`

In [ ]:
# ШАГ 0 — Установка библиотек
!pip install pandas openpyxl --quiet
print('✅ Готово')

In [ ]:
# ШАГ 1 — Загрузка Excel-файла
from google.colab import files
print('📂 Выберите .xlsx файл с таблицей отключений')
uploaded = files.upload()
xlsx_filename = list(uploaded.keys())[0]
print(f'\n✅ Загружен: {xlsx_filename}')

In [ ]:
# ШАГ 2 — Конвертация с нормализацией и дедупликацией
import pandas as pd
import json
import re

def format_date(val):
    if pd.isna(val) or val == '' or val is None:
        return ''
    if hasattr(val, 'strftime'):
        return val.strftime('%d.%m.%Y')
    return str(val).strip()

def format_period(val):
    """Нормализует строку с диапазоном дат к виду dd.mm.yyyy-dd.mm.yyyy.
    Примеры:
      '25.07-04.08..2025' -> '25.07.2025-04.08.2025'
      '23-24.07.2025'     -> '23.07.2025-24.07.2025'
      '12.07.-21.07.2025' -> '12.07.2025-21.07.2025'
      '27.05.2025'        -> '27.05.2025'
    """
    if pd.isna(val) or val == '' or val is None:
        return ''
    if hasattr(val, 'strftime'):
        return val.strftime('%d.%m.%Y')
    s = str(val).strip()
    s = re.sub(r'\.{2,}', '.', s)   # двойные точки
    s = re.sub(r'\.\-', '-', s)     # '12.07.-21' -> '12.07-21'
    s = s.strip('.')
    def pad(x): return f'{int(x):02d}'
    # dd.mm.yyyy-dd.mm.yyyy (уже нормально)
    m = re.match(r'^(\d{1,2})\.(\d{1,2})\.(\d{4})[-\u2013](\d{1,2})\.(\d{1,2})\.(\d{4})$', s)
    if m:
        d1,mo1,y1,d2,mo2,y2 = m.groups()
        return f'{pad(d1)}.{pad(mo1)}.{y1}-{pad(d2)}.{pad(mo2)}.{y2}'
    # dd.mm-dd.mm.yyyy
    m = re.match(r'^(\d{1,2})\.(\d{1,2})-(\d{1,2})\.(\d{1,2})\.(\d{4})$', s)
    if m:
        d1,mo1,d2,mo2,y = m.groups()
        return f'{pad(d1)}.{pad(mo1)}.{y}-{pad(d2)}.{pad(mo2)}.{y}'
    # dd-dd.mm.yyyy
    m = re.match(r'^(\d{1,2})-(\d{1,2})\.(\d{1,2})\.(\d{4})$', s)
    if m:
        d1,d2,mo,y = m.groups()
        return f'{pad(d1)}.{pad(mo)}.{y}-{pad(d2)}.{pad(mo)}.{y}'
    # dd.mm.yyyy
    m = re.match(r'^(\d{1,2})\.(\d{1,2})\.(\d{4})$', s)
    if m:
        d,mo,y = m.groups()
        return f'{pad(d)}.{pad(mo)}.{y}'
    return s

# Синонимы типов улиц → канонический вид
STREET_SYNONYMS = [
    (r'\bпр-кт\.?(?=\s|,|$)',    'проспект'),
    (r'\bпр-т\.?(?=\s|,|$)',     'проспект'),
    (r'\bпроспект(?=\s|,|$)',    'проспект'),
    (r'\bпр-д\.?(?=\s|,|$)',     'проезд'),
    (r'\bпроезд(?=\s|,|$)',      'проезд'),
    (r'\bул\.?(?=\s|,|$)',       'ул.'),
    (r'\bулица(?=\s|,|$)',       'ул.'),
    (r'\bпер\.?(?=\s|,|$)',      'пер.'),
    (r'\bпереулок(?=\s|,|$)',    'пер.'),
    (r'\bбул\.?(?=\s|,|$)',      'бул.'),
    (r'\bбульвар(?=\s|,|$)',     'бул.'),
    (r'\bпл\.?(?=\s|,|$)',       'пл.'),
    (r'\bплощадь(?=\s|,|$)',     'пл.'),
    (r'\bст\.?(?=\s|,|$)',       'станция'),
    # набережная — разворачиваем аббревиатуру, НЕ выносим в префикс
    (r'\bнаб\.?(?=\s|,|$)',      'набережная'),
]

# Населённые пункты — отдельный словарь, не типы улиц
LOCALITY_SYNONYMS = [
    (r'\bп\.?(?=\s|,|$)',        'посёлок'),
    (r'\bпос\.?(?=\s|,|$)',      'посёлок'),
    (r'\bпосёлок(?=\s|,|$)',     'посёлок'),
    (r'\bпоселок(?=\s|,|$)',     'посёлок'),
]
LOCALITY_TYPES  = {'посёлок', 'станция', 'деревня', 'село', 'микрорайон'}
STREET_PREFIXES = {'проспект', 'проезд', 'ул.', 'пер.', 'бул.', 'пл.', 'шоссе', 'тупик', 'аллея'}

# DEDUP_IGNORE: все служебные слова + их варианты без точек
_raw = STREET_PREFIXES | LOCALITY_TYPES | {'набережная', 'д.', 'д'}
DEDUP_IGNORE = _raw | {re.sub(r'\.', '', t) for t in _raw}

def expand_bolshaya(s):
    """Б. Октябрьская / Б. ДОНСКАЯ → Большая Октябрьская / Большая Донская."""
    def repl(m):
        word = m.group(1)
        return 'Большая ' + word[0].upper() + word[1:].lower()
    s = re.sub(r'\bБ\.\s+([А-ЯЁа-яёA-Za-z]+)', repl, s)
    s = re.sub(r'\bБ\b\s+([А-ЯЁ]{2,})',         repl, s)  # ДОНСКАЯ и подобные
    return s

def normalize_address(addr):
    s = str(addr).strip()
    # 1. Город
    s = re.sub(r'^г\.?\s*Ярославль[,\s]+', '', s, flags=re.IGNORECASE)
    # 2. Б. → Большая
    s = expand_bolshaya(s)
    # 3. Пробел после слитного ул.Название
    s = re.sub(r'\bул\.([А-ЯЁа-яё])', r'ул. \1', s)
    # 4. Населённые пункты
    for p, r in LOCALITY_SYNONYMS:
        s = re.sub(p, r, s, flags=re.IGNORECASE)
    # 5. Типы улиц
    for p, r in STREET_SYNONYMS:
        s = re.sub(p, r, s, flags=re.IGNORECASE)
    # 6. Повторный пробел после ул.
    s = re.sub(r'ул\.([А-ЯЁа-яё])', r'ул. \1', s)
    # 7. Набережная с маленькой буквы внутри
    s = re.sub(r'(?<=\s)Набережная\b', 'набережная', s)
    # 8. Дом и корпус
    s = re.sub(r'\bд\.?\s*(\d)', r'д. \1', s)
    s = re.sub(r'\bкорп?\.?\s*(\d)', r'корп. \1', s)
    # 9. Число в конце без д.
    s = re.sub(r'(?<!\.)\s+(\d[\d/а-яА-ЯёЁ]*)$', r' д. \1', s)
    # 10. Запятые и пробелы
    s = re.sub(r'\s*,\s*', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    # 11. Реструктурируем адрес с населённым пунктом внутри
    #     'ул. посёлок Дубки Спортивная д. 9' → 'посёлок Дубки, ул. Спортивная д. 9'
    locality_re = '|'.join(re.escape(lt) for lt in LOCALITY_TYPES)
    m = re.match(
        r'^(ул\.)\s+(' + locality_re + r')\s+(\S+)\s+(.+?)(\s+д\..*)$',
        s, flags=re.IGNORECASE
    )
    if m:
        s = f'{m.group(2)} {m.group(3)}, {m.group(1)} {m.group(4)}{m.group(5)}'
    # 12. Тип-префикс улицы в начало (не трогаем если начинается с нас. пункта)
    tokens = s.split()
    first = tokens[0].lower().rstrip(',') if tokens else ''
    if first not in LOCALITY_TYPES:
        type_idx = next((i for i, t in enumerate(tokens) if t.lower() in STREET_PREFIXES), None)
        if type_idx is not None and type_idx != 0:
            w = tokens.pop(type_idx)
            tokens.insert(0, w)
            s = ' '.join(tokens)
    # 13. Убираем дублирующийся тип-префикс
    tokens, seen, clean = s.split(), False, []
    for t in tokens:
        if t.lower() in STREET_PREFIXES:
            if not seen: clean.append(t); seen = True
        else: clean.append(t)
    s = ' '.join(clean)
    # 14. Набережная после префикса → перед д.
    prefixes_re = '|'.join(re.escape(p) for p in STREET_PREFIXES)
    s = re.sub(
        r'^(' + prefixes_re + r')\s+набережная\s+(.+?)(\s+д\..*)',
        r'\1 \2 набережная\3', s, flags=re.IGNORECASE
    )
    return s

def dedup_key(address_normalized, repair, hydro1, hydro2, source):
    """Ключ: служебные слова убраны (с точками и без), токены отсортированы."""
    tokens = address_normalized.lower().split()
    tokens = [re.sub(r'[,.]', '', t) for t in tokens]  # убираем пунктуацию
    tokens = [t for t in tokens if t and t not in DEDUP_IGNORE]
    return (' '.join(sorted(tokens)), repair, hydro1, hydro2, source)

def convert(xlsx_path):
    xl = pd.ExcelFile(xlsx_path)
    raw_records = []

    for sheet_name in xl.sheet_names:
        df = pd.read_excel(xl, sheet_name=sheet_name, header=None, skiprows=2)
        for _, row in df.iterrows():
            address_raw = str(row.iloc[3]).strip() if pd.notna(row.iloc[3]) else ''
            if not address_raw or address_raw == 'nan':
                continue

            source = str(row.iloc[10]).strip() if pd.notna(row.iloc[10]) else ''

            raw_records.append({
                'address':   normalize_address(address_raw),
                'repair':    format_period(row.iloc[4]),
                'repair_on': format_date(row.iloc[5]),
                'hydro1':    format_period(row.iloc[6]),
                'hydro1_on': format_date(row.iloc[7]),
                'hydro2':    format_period(row.iloc[8]),
                'hydro2_on': format_date(row.iloc[9]),
                'source':    source,
            })

    # Дедупликация: ключ нормализован в нижний регистр → ловим 'ул.' vs 'ул' и т.п.
    seen = set()
    unique = []
    for r in raw_records:
        key = dedup_key(r['address'], r['repair'], r['hydro1'], r['hydro2'], r['source'])
        if key not in seen:
            seen.add(key)
            unique.append(r)

    return raw_records, unique

print(f'⚙️  Обрабатываю: {xlsx_filename}\n')
raw, records = convert(xlsx_filename)

print(f'Исходных строк:     {len(raw)}')
print(f'После дедупликации: {len(records)}')
print(f'Удалено дублей:     {len(raw) - len(records)}')

with open('data.json', 'w', encoding='utf-8') as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

print(f'\n✅ Сохранено: data.json')

In [ ]:
# ШАГ 3 — Скачать data.json
from google.colab import files
files.download('data.json')
print('⬇️  Скачан data.json')